# S02 · Fit a straight line to data

Back to the property website. It has the sizes and prices of past flats, and it
wants to price a new flat that nobody has listed yet. We draw the single best
straight line through the past sales, then read a price off that line. This is
where the array skills from the first two notebooks pay off.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press play on each cell, top
  to bottom, and read the plain-English note above each one.
- New to the idea of fitting a line? Open `primers/vectors_and_matrices.md`, and
  there is a whole primer on the idea in `primers/least_squares_and_lines.md`.
- The main path is Steps 1 to 6, and it uses one ready-made solver. The
  **Stretch (optional)** cell at the end does the same fit by hand for those who
  want the linear algebra. Skipping it costs you nothing.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses numpy and matplotlib, both of which Colab already ships.
# So there is nothing to install here.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np                 # fast maths on lists of numbers
import matplotlib.pyplot as plt      # drawing charts

# A seed so we all get the same random data and the same result.
np.random.seed(0)
print("Ready.")

## Step 1 — make some flats we understand

The safest way to trust a method is to try it where we already know the answer. So
we invent 30 flats ourselves. We decide the real market rule, then add a little
random wobble so the prices are not perfectly neat, just like real life.

Our secret rule: price rises by about 0.05 lakh for every extra square foot,
starting from a base of 10 lakh.

In [ ]:
# 30 flat sizes, evenly spread from 400 to 1400 square feet.
flat_size = np.linspace(400, 1400, 30)

# The true rule we are pretending the market follows, plus some random wobble.
noise = np.random.normal(0, 4.0, size=30)          # a few lakh up or down
flat_price = 0.05 * flat_size + 10.0 + noise

print("first 5 sizes  (sq ft):", flat_size[:5].round(0))
print("first 5 prices (lakh) :", flat_price[:5].round(1))

## Step 2 — always look at the data first

Before fitting anything, plot the data and just look. Each dot is one flat: its
size across the bottom, its price up the side. The prices should drift upward as
size grows.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(flat_size, flat_price, color="#2E75B6")
plt.xlabel("flat size (sq ft)")
plt.ylabel("price (lakh Rs)")
plt.title("Past flat sales")
plt.show()

## Step 3 — set up the data for the line

A straight line is `price = slope * size + starting_value`. To let the solver find
both the slope and the starting value together, we lay the data out as a table `X`
with two columns: the first column is the sizes, the second is all ones. That
column of ones is what carries the starting value (the height of the line where
size is zero).

In [ ]:
# The first column is the sizes; the second column is all ones.
column_of_sizes = flat_size
column_of_ones = np.ones(30)

# Stack the two columns side by side into a table with shape (30, 2).
X = np.column_stack([column_of_sizes, column_of_ones])

print("X shape:", X.shape)
print()
print("first 3 rows of X (size, 1):")
print(X[0:3, :])

## Step 4 — let NumPy find the best line

Now the one idea. Out of all possible straight lines, we want the one whose total
miss from the dots is smallest. NumPy has a ready-made solver for exactly this,
`np.linalg.lstsq` (least squares). We hand it the table `X` and the prices, and it
hands back the slope and the starting value in one call.

In [ ]:
# lstsq returns several things; the first is the pair of numbers we want.
# rcond=None just picks a sensible default; you can ignore it for now.
solver_output = np.linalg.lstsq(X, flat_price, rcond=None)
weights = solver_output[0]

slope = weights[0]
starting_value = weights[1]

print("slope          (true value 0.05):", round(slope, 4), "lakh per sq ft")
print("starting value (true value 10.0):", round(starting_value, 2), "lakh")

## Step 5 — predict the price of a new flat

This is the payoff. A new 1000 sq ft flat comes to the website. We read the line at
1000 and out comes an estimated price. This is exactly what a property site shows
the moment someone lists a flat.

In [ ]:
new_flat_size = 1000
predicted_price = slope * new_flat_size + starting_value

print("A", new_flat_size, "sq ft flat is worth about",
      round(predicted_price, 1), "lakh.")

## Step 6 — draw the fitted line

Let us put the line on top of the dots and see how snugly it fits. We compute the
line's predicted price at every size and draw it through the cloud of points.

In [ ]:
# The line's predicted price for each of our flats.
predicted_line = slope * flat_size + starting_value

plt.figure(figsize=(7, 5))
plt.scatter(flat_size, flat_price, color="#2E75B6", label="real sales")
plt.plot(flat_size, predicted_line, color="#C0392B", linewidth=3,
         label="the best-fit line")
plt.xlabel("flat size (sq ft)")
plt.ylabel("price (lakh Rs)")
plt.title("The best-fit line through the flats")
plt.legend()
plt.show()

### Stretch (optional) — find the same line by hand

Skip this if you are new to code or to matrices. If you know some linear algebra,
here is the satisfying part. The same best line comes straight out of a one-line
formula called the **normal equation**, `(Xᵀ X) w = Xᵀ y`, with no special solver.
We build both sides with `@` (matrix multiply) and `.T` (transpose), solve with
`np.linalg.solve`, and check it matches `np.linalg.lstsq` to the last decimal.

In [ ]:
# Build the two sides of the normal equation.
# .T is the transpose; @ is matrix multiply.
XtX = X.T @ X
Xty = X.T @ flat_price

# Solve for the weights. We use solve, not an inverse - it is the safe way.
weights_by_hand = np.linalg.solve(XtX, Xty)

print("by hand :", weights_by_hand.round(4))
print("lstsq   :", weights.round(4))
print()
print("They match?", np.allclose(weights_by_hand, weights))
print("The ready-made solver was doing this same linear algebra all along.")

## What you just did

You took past flat sales, laid them out as a table, drew the single best straight
line through them, and used that line to price a brand-new flat. That is a complete,
real prediction machine, built out of the array skills from the first two notebooks.

This same least-squares idea comes back in full in Session 7, where you judge such a
model honestly by testing it on flats it was never shown. Today you saw it is, at
heart, a little arithmetic on lists and tables of numbers.